# OpenOBB: train trên Colab

Chạy lần lượt từng ô. GPU: *Runtime → Change runtime type* → **G4** (High-RAM).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài OpenOBB

Luôn lấy bản mới nhất trên GitHub.

In [ ]:
!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/OpenOBB.git

## 3. Giải nén dataset

Bỏ `#` ở dòng của dataset muốn dùng, thêm `#` vào các dòng còn lại. Chỉ giải nén **một** dataset vào `/content/dataset_local`.
Nếu có nhiều dataset ở đó, ô sau sẽ chọn `data.yaml` nông nhất, có thể là của dataset khác.

In [ ]:
# !unzip -q -o "/content/drive/MyDrive/Dataset/tu_ph/Wall_Color/data/dataset_wall_color_v3.zip" -d /content/dataset_local
# !unzip -q -o "/content/drive/MyDrive/Dataset/tu_ph/Tivi/data_tivi_v2.zip" -d /content/dataset_local
# !unzip -q -o "/content/drive/MyDrive/Dataset/tu_ph/Door_Foot/data_train.zip" -d /content/dataset_local
!unzip -q "/content/drive/MyDrive/Dataset/Floor_plan/dataset_obb_train_v3.zip" -d /content/dataset_local


## 4. Tạo `/content/data.yaml`

Ô này tìm `data.yaml` trong dataset đã giải nén, rồi ghi lại với `path` tuyệt đối.

In [ ]:
import glob, yaml; y = min(glob.glob("/content/dataset_local/**/data.yaml", recursive=True), key=len)
yaml.safe_dump({**yaml.safe_load(open(y)), "path": y.rsplit("/", 1)[0]}, open("/content/data.yaml", "w"))

## 5. Train

- `model=`: tên kiến trúc (ví dụ `openobb23x`) để train mới, hoặc đường dẫn `.pt` để fine-tune tiếp
  (checkpoint trên Drive, hoặc `best.pt` của run trước).
- Mỗi lần train tạo một thư mục mới: `runs/obb/train`, `train2`...
- `lr=` là dùng đúng giá trị đó. Bỏ `lr` thì tự đặt `0.002·5/(4+nc)` như YOLO.
- `compile=true` đã là mặc định cho v18–v24.
- `rotate_p` mặc định là 0 (giống `degrees=0` của YOLO). Bỏ hẳn `rotate_p=0.5` nếu thấy box vật dài lệch hướng (F110).
- Mất phiên giữa chừng: chạy lại ô 1–4 rồi chạy `!openobb train resume=True`.

In [ ]:
!openobb train \
data=/content/data.yaml \
model=/content/runs/obb/train/weights/best.pt \
epochs=30 \
imgsz=960 \
batch=32 \
rotate_p=0.5 \
lr=0.001 \
compile=true

# /content/drive/MyDrive/Train/tu_ph/Wall_Color/model/openobb23x_new.pt
# /content/runs/obb/train/weights/best.pt


## 6. Tải `best.pt` của run mới nhất về máy

In [ ]:
import os
import glob
from google.colab import files

# Tìm tất cả các thư mục train trong runs/obb/
train_dirs = glob.glob('runs/obb/train*')

if train_dirs:
    # Tìm thư mục mới nhất dựa trên thời gian sửa đổi (mtime)
    latest_train_dir = max(train_dirs, key=os.path.getmtime)
    print(f"Thư mục train mới nhất được phát hiện: {latest_train_dir}")

    model_path = os.path.join(latest_train_dir, 'weights', 'best.pt')

    if os.path.exists(model_path):
        try:
            files.download(model_path)
            print(f"Đang chuẩn bị tải file từ: {model_path} ...")
        except Exception as e:
            print(f"Lỗi khi tải file: {e}")
    else:
        print(f"Không tìm thấy file best.pt tại: {model_path}")
else:
    print("Không tìm thấy bất kỳ thư mục train nào trong runs/obb/")